# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa, Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey dataset from Kilifi County, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Display key information
metadata = dataset.metadata
print("Dataset Name:", metadata.name)
print("Description:\n", metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We use the dataset metadata to enumerate available record sets, fields, and columns. All are referenced by their `@id` values.

In [ ]:
# List all available record sets by their @id
recordset_objs = getattr(metadata, 'record_sets', metadata.record_sets if hasattr(metadata, 'record_sets') else metadata.recordSet)
if callable(recordset_objs):  # for future compatibility
    recordset_objs = recordset_objs()
    
# If `recordset_objs` is empty, try .recordSet (some Croissant metadata uses .recordSet)
if not recordset_objs:
    recordset_objs = getattr(metadata, 'recordSet', [])

record_sets = []
print('Record Sets:')
for rs in recordset_objs:
    rs_id = getattr(rs, '@id', None) or rs.get('@id') if hasattr(rs, 'get') else None
    print(f" - {rs_id} | name: {getattr(rs, 'name', getattr(rs, 'label', None))}")
    record_sets.append(rs_id)

# Print fields for the first record set
if recordset_objs:
    sample_recordset = recordset_objs[0]
    print(f"\nFields in record set {getattr(sample_recordset, '@id', None)}:")
    fields = getattr(sample_recordset, 'fields', None) or getattr(sample_recordset, 'field', None)
    if fields is not None:
        for f in fields:
            print(f" - {getattr(f, '@id', None)} | name: {getattr(f, 'name', getattr(f, 'label', None))}")
    else:
        print("  No fields found.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Each record set is referenced by its `@id`.

In [ ]:
# Determine the record set @ids
from pprint import pprint

# For this dataset, the record sets may need to be manually specified if not listed in metadata (common in e.g. OpenDataArticle style datasets)
if not record_sets:
    # Try to get all the @id for plausible record sets
    # mlcroissant falls back to .recordSet for legacy or alternate vocab
    record_sets = []
    if hasattr(metadata, 'recordSet'):
        for rs in metadata.recordSet:
            rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else (rs if isinstance(rs, str) else None)
            record_sets.append(rs_id)

# If still empty, provide an example (for this dataset, record_set is likely 'cr:RecordSet:main')
if not record_sets:
    # This is a typical default for mlcroissant-compliant record set IDs
    record_sets = ['cr:RecordSet:main']

# Extract data from each record set
dataframes = {}
for record_set in record_sets:
    print(f"Loading records from record set: {record_set}")
    records = list(dataset.records(record_set=record_set))
    if records:
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records. Fields: {list(df.columns)}")
        dataframes[record_set] = df
    else:
        print(f"No records found for record set {record_set}.")

# Show first rows and field names for the first DataFrame
if dataframes:
    main_rs = record_sets[0]
    print(f"Columns in {main_rs}: ")
    pprint(list(dataframes[main_rs].columns))
    dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll choose a relevant numeric field, e.g., a GAD-7, PHQ-9, or PSQ score field, referring to each field by its `@id` if columns use them as names. 

Replace `<numeric_field_id>` and `<group_field>` with the actual field `@id`s found in the previous section.

In [ ]:
import numpy as np

# Choose which record set to analyze (use the main one if multiple are present)
record_set_id = record_sets[0]
df = dataframes[record_set_id]

# Try to infer a numeric field for analysis
numeric_field_candidates = [c for c in df.columns if 'phq' in c.lower() or 'gad' in c.lower() or 'psq' in c.lower() or c.endswith('_score')]
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
else:
    # Fallback: simply use a column that looks numeric
    numeric_field = df.select_dtypes(include=np.number).columns[0]
print(f"Selected numeric field for analysis: {numeric_field}")

threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt a group by a demographic field (e.g. 'age', 'gender', or similar; use the column @id or name as appropriate)
group_candidates = [c for c in df.columns if 'gender' in c.lower() or 'age_group' in c.lower() or 'marital' in c.lower() or 'education' in c.lower()]
if group_candidates:
    group_field = group_candidates[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    group_field = None
    print("No obvious group field found to group by.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot the distribution of the chosen numeric field and, if grouping is available, compare means across the grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we explored the Kilifi County Mental Health Survey dataset using `mlcroissant`.
- We loaded the dataset via its public Croissant schema and reviewed record sets and fields using their `@id`s.
- Data for the main record set was loaded into pandas for programmatic inspection.
- We identified a key numeric mental health indicator (GAD-7/PHQ-9/PSQ score), performed basic filtering, normalization, and group analysis by a demographic attribute.
- Data visualizations revealed key distributions, and groupings highlighted demographic patterns.

This demonstrates the reproducible, FAIR, and transparent data access and analysis workflow supported by `mlcroissant` and well-documented Croissant schemas.

For further analysis, you may connect this workflow to machine learning pipelines, statistical testing, or integrate new Croissant record sets as the dataset grows.

_Notebook generated for the Senscience FAIR² Mental Health Survey Example._